# Set up the Lakehouse target table (explained)

Attach `lh_meridian_hr` as the default lakehouse, then run this notebook before the event-ingestion pipeline. The DDL is idempotent and safely pre-creates the shared Delta table before parallel Copy activities append to it.

> This is an annotated copy of `nb_00_setup_lakehouse.ipynb`. Each code cell is preceded by a **Summary** and a collapsible **line-by-line** explanation.

## Create the schema and pre-create the target table

**Summary.** Ensure the `bronze` schema exists and pre-create the empty `bronze.workforce_events_raw` Delta table with an explicit column layout, so the pipeline's parallel Copy activities append to a table that already exists instead of racing to create it.

<details>
<summary>Line-by-line details</summary>

- `spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")` — creates the `bronze` schema (database) if it is not already present; harmless on re-runs.
- `CREATE TABLE IF NOT EXISTS bronze.workforce_events_raw (...)` — defines the raw events table only when it is missing, which makes the DDL idempotent (safe to run repeatedly).
- The column list fixes the schema up front: identifiers (`event_id`, `employee_id`, `cost_center_id`), the `event_date`, classification (`classification_group`, `classification_level`), the `event_type`, money (`amount_local` as `DECIMAL(18, 2)` plus `local_currency`), `work_country_code`, and lineage columns (`ingest_ts`, `source_system`).
- `USING DELTA` — stores the table as a Delta Lake table (Parquet files plus a transaction log), so appends are ACID.
- `print("bronze.workforce_events_raw is ready")` — writes a confirmation line to the cell output.

</details>

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.workforce_events_raw (
    event_id STRING,
    event_date DATE,
    employee_id STRING,
    cost_center_id STRING,
    classification_group STRING,
    classification_level INT,
    event_type STRING,
    amount_local DECIMAL(18, 2),
    local_currency STRING,
    work_country_code STRING,
    ingest_ts TIMESTAMP,
    source_system STRING
)
USING DELTA
""")

print("bronze.workforce_events_raw is ready")